In [1]:
import duckdb
con = duckdb.connect()
path = "../data/raw/amazon_reviews/Toys_and_Games.jsonl.gz"
con.sql(f"SELECT * FROM read_json('{path}', format='newline_delimited') LIMIT 5").df()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Granddaughters love them!,I purchased several of these for my granddaugh...,[],B09QH7QJS7,B09QH7QJS7,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1677939664713,0,True
1,3.0,Arrived broken,"It’s cute, but it arrived broken & has some pr...",[{'small_image_url': 'https://m.media-amazon.c...,B06XYKSKQP,B06XYKSKQP,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1639855230760,1,True
2,5.0,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,[],B07SFF3YQW,B07XRSD5R9,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1580949719154,0,True
3,5.0,Janod quality and very cute...,This was great for my daughters circus themed ...,[],B007JWWUDW,B007JWWUDW,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1471542588000,0,True
4,3.0,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,[],B00MZG6OO8,B00MZG6OO8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1471541996000,2,True


In [2]:
con.sql(f"SELECT count(*) FROM read_json('{path}', format='newline_delimited', columns={{'asin':'VARCHAR'}})").fetchall()

[(16260406,)]

In [3]:
purchase_path = "../data/raw/open_ecommerce/amazon-purchases.csv"
categories = ['TOILET_PAPER', 'SURFACE_CLEANING_WIPE', 'PUZZLES',
              'BAKING_PAN', 'BLANKET', 'DRINKING_CUP']

asin_list = con.sql(f"""
    SELECT
        upper(trim("ASIN/ISBN (Product Code)")) AS asin,   -- 공백 제거, 대문자 통일
        Category AS product_type,
        count(*) AS purchase_rows                          -- ASIN별 구매 행 수
    FROM read_csv('{purchase_path}', all_varchar=true)
    WHERE Category IN {tuple(categories)}
      AND "ASIN/ISBN (Product Code)" IS NOT NULL           -- ASIN 결측 제외
      AND "Order Date" BETWEEN '2018-01-01' AND '2022-12-31'
    GROUP BY 1, 2
    ORDER BY 2, 3 DESC
""").df()

print(asin_list.groupby('product_type').size())
asin_list.to_csv("../data/interim/cleaned_purchase/asin_list_temp.csv", index=False)


product_type
BAKING_PAN               1070
BLANKET                  4243
DRINKING_CUP             5626
PUZZLES                  2283
SURFACE_CLEANING_WIPE     244
TOILET_PAPER              352
dtype: int64


In [4]:
asin_map = con.sql("""
    SELECT asin, arg_max(product_type, purchase_rows) AS product_type
    FROM asin_list
    GROUP BY asin
""").df()
print(len(asin_map))   # 13817 이 나와야 해요

13817


In [7]:
con.sql("SET TimeZone = 'UTC'")
cols = "{'asin':'VARCHAR','parent_asin':'VARCHAR','rating':'DOUBLE','title':'VARCHAR','text':'VARCHAR','timestamp':'BIGINT','helpful_vote':'BIGINT','verified_purchase':'BOOLEAN','user_id':'VARCHAR'}"
con.sql("SET preserve_insertion_order = false")   # 메모리 절약

for f in ["Toys_and_Games", "Health_and_Household", "Home_and_Kitchen"]:
    path = f"../data/raw/amazon_reviews/{f}.jsonl.gz"
    out = f"../data/interim/filtered_reviews/{f}_filtered.parquet"
    con.sql(f"""
        COPY (
            SELECT r.*, m.product_type, '{f}' AS source_file,
                   CAST(to_timestamp(r.timestamp / 1000) AS DATE) AS review_date
            FROM read_json('{path}', format='newline_delimited', columns={cols}) r
            JOIN asin_map m ON r.asin = m.asin
            WHERE to_timestamp(r.timestamp / 1000) >= TIMESTAMP '2018-01-01'
              AND to_timestamp(r.timestamp / 1000) <  TIMESTAMP '2023-01-01'
        ) TO '{out}' (FORMAT parquet)
    """)
    print(f, con.sql(f"SELECT product_type, count(*) FROM '{out}' GROUP BY 1 ORDER BY 1").fetchall())

Toys_and_Games [('BAKING_PAN', 650), ('BLANKET', 833), ('DRINKING_CUP', 2903), ('PUZZLES', 175581), ('TOILET_PAPER', 38)]
Health_and_Household [('BLANKET', 31), ('DRINKING_CUP', 34213), ('PUZZLES', 432), ('SURFACE_CLEANING_WIPE', 53968), ('TOILET_PAPER', 102233)]
Home_and_Kitchen [('BAKING_PAN', 138009), ('BLANKET', 250013), ('DRINKING_CUP', 358912), ('PUZZLES', 59), ('SURFACE_CLEANING_WIPE', 16), ('TOILET_PAPER', 5)]


In [8]:
files = "['../data/interim/filtered_reviews/Toys_and_Games_filtered.parquet', '../data/interim/filtered_reviews/Health_and_Household_filtered.parquet', '../data/interim/filtered_reviews/Home_and_Kitchen_filtered.parquet']"
steps = {}

steps['1_합친 리뷰'] = con.sql(f"SELECT count(*) FROM read_parquet({files})").fetchone()[0]

# rating 또는 timestamp가 없으면 삭제
con.sql(f"""CREATE OR REPLACE TABLE reviews AS
    SELECT * FROM read_parquet({files})
    WHERE rating IS NOT NULL AND timestamp IS NOT NULL""")
steps['2_결측 제거 후'] = con.sql("SELECT count(*) FROM reviews").fetchone()[0]

# 같은 사람·같은 상품·같은 시각·같은 내용이면 하나만 남김
con.sql("""CREATE OR REPLACE TABLE reviews AS
    SELECT * FROM reviews
    QUALIFY row_number() OVER (PARTITION BY user_id, asin, timestamp, text ORDER BY source_file) = 1""")
steps['3_중복 제거 후'] = con.sql("SELECT count(*) FROM reviews").fetchone()[0]
print(steps)

# 품목별 요약 (전처리 로그·발표용)
print(con.sql("""SELECT product_type, count(*) AS reviews, count(DISTINCT asin) AS asins,
    round(avg(rating), 2) AS avg_rating,
    sum(CASE WHEN text IS NULL OR trim(text) = '' THEN 1 ELSE 0 END) AS empty_text
    FROM reviews GROUP BY 1 ORDER BY 1""").df())

con.sql("COPY reviews TO '../data/interim/filtered_reviews/reviews_filtered.parquet' (FORMAT parquet)")

{'1_합친 리뷰': 1117896, '2_결측 제거 후': 1117896, '3_중복 제거 후': 1107639}
            product_type  reviews  asins  avg_rating  empty_text
0             BAKING_PAN   137451   1007        4.39       183.0
1                BLANKET   248588   2727        4.33       244.0
2           DRINKING_CUP   392142   4238        4.34       544.0
3                PUZZLES   174324   2019        4.17       296.0
4  SURFACE_CLEANING_WIPE    53538    190        4.25        37.0
5           TOILET_PAPER   101596    322        4.04        54.0


In [9]:
con.sql("SET TimeZone = 'UTC'")
cols = "{'asin':'VARCHAR','parent_asin':'VARCHAR','rating':'DOUBLE','title':'VARCHAR','text':'VARCHAR','timestamp':'BIGINT','helpful_vote':'BIGINT','verified_purchase':'BOOLEAN','user_id':'VARCHAR'}"
con.sql("SET preserve_insertion_order = false")   # 메모리 절약

for f in ["Toys_and_Games", "Health_and_Household", "Home_and_Kitchen"]:
    path = f"../data/raw/amazon_reviews/{f}.jsonl.gz"
    out = f"../data/interim/filtered_reviews/{f}_filtered.parquet"
    con.sql(f"""
        COPY (
            SELECT r.*, m.product_type, '{f}' AS source_file,
                   CAST(to_timestamp(r.timestamp / 1000) AS DATE) AS review_date
            FROM read_json('{path}', format='newline_delimited', columns={cols}) r
            JOIN asin_map m ON r.asin = m.asin
            WHERE to_timestamp(r.timestamp / 1000) >= TIMESTAMP '2018-01-01'
              AND to_timestamp(r.timestamp / 1000) <  TIMESTAMP '2023-01-01'
        ) TO '{out}' (FORMAT parquet)
    """)
    print(f, con.sql(f"SELECT product_type, count(*) FROM '{out}' GROUP BY 1 ORDER BY 1").fetchall())

Toys_and_Games [('BAKING_PAN', 650), ('BLANKET', 833), ('DRINKING_CUP', 2903), ('PUZZLES', 175581), ('TOILET_PAPER', 38)]
Health_and_Household [('BLANKET', 31), ('DRINKING_CUP', 34213), ('PUZZLES', 432), ('SURFACE_CLEANING_WIPE', 53968), ('TOILET_PAPER', 102233)]
Home_and_Kitchen [('BAKING_PAN', 138009), ('BLANKET', 250013), ('DRINKING_CUP', 358912), ('PUZZLES', 59), ('SURFACE_CLEANING_WIPE', 16), ('TOILET_PAPER', 5)]


In [10]:
weekly = con.sql("""
WITH weeks AS (   -- 2017-12-31(일) ~ 2022-12-25(일), 261주
    SELECT CAST(w AS DATE) AS week
    FROM generate_series(DATE '2017-12-31', DATE '2022-12-25', INTERVAL 7 DAY) g(w)
),
types AS (SELECT DISTINCT product_type FROM reviews),
agg AS (
    SELECT product_type,
           -- 일요일 시작 주: 하루 더해서 월요일 기준으로 자른 뒤 하루 빼기
           CAST(date_trunc('week', review_date + INTERVAL 1 DAY) - INTERVAL 1 DAY AS DATE) AS week,
           count(*) AS review_count,
           avg(rating) AS avg_rating,
           avg(CASE WHEN rating = 1 THEN 1.0 ELSE 0 END) AS share_1star,
           avg(CASE WHEN rating = 5 THEN 1.0 ELSE 0 END) AS share_5star,
           avg(CASE WHEN verified_purchase THEN 1.0 ELSE 0 END) AS verified_share
    FROM reviews GROUP BY 1, 2
)
SELECT w.week, t.product_type,
       coalesce(a.review_count, 0) AS review_count,   -- 리뷰 없는 주는 0
       round(a.avg_rating, 3) AS avg_rating,          -- 리뷰 없는 주는 비워 둠
       round(a.share_1star, 3) AS share_1star,
       round(a.share_5star, 3) AS share_5star,
       round(a.verified_share, 3) AS verified_share
FROM weeks w CROSS JOIN types t
LEFT JOIN agg a ON a.week = w.week AND a.product_type = t.product_type
ORDER BY t.product_type, w.week
""").df()

print(len(weekly), weekly['week'].nunique())   # 1566, 261 이 나와야 해요
print(weekly.groupby('product_type')['review_count'].sum())
weekly.to_csv("../data/processed/weekly_reviews/weekly_reviews.csv", index=False)

1566 261
product_type
BAKING_PAN               137451
BLANKET                  248588
DRINKING_CUP             392142
PUZZLES                  174324
SURFACE_CLEANING_WIPE     53538
TOILET_PAPER             101596
Name: review_count, dtype: int64
